# Module 8.2: Machine Learning Connections to Factor Models

## Learning Objectives

By completing this notebook, you will:

1. Understand the mathematical equivalence between linear autoencoders and PCA
2. Implement deep autoencoders for nonlinear factor extraction
3. Build variational autoencoders (VAE) for probabilistic factor models with uncertainty
4. Compare linear vs nonlinear approaches on synthetic and realistic data
5. Evaluate trade-offs between interpretability, theory, and predictive performance

## Prerequisites

- Principal Component Analysis (Module 1)
- Factor model estimation (Module 3)
- Basic neural network concepts
- PyTorch fundamentals

## Estimated Time: 90 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Principal Component Analysis (Module 1)",
    "Factor model estimation (Module 3)",
    "Basic neural network concepts",
    "PyTorch fundamentals"
])

## Setup and Imports

We'll use PyTorch for neural network implementations and scikit-learn for traditional methods.

In [ ]:
# Purpose: Import libraries for ML-based factor models
# Key Concept: Neural networks as nonlinear factor extractors

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# PyTorch for neural networks
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

# Check if GPU available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

## 1. Linear Autoencoders and PCA Equivalence

### Mathematical Connection

**Linear Autoencoder**:
$$\text{Encoder: } F_t = W_1^T X_t$$
$$\text{Decoder: } \hat{X}_t = W_2 F_t$$
$$\min_{W_1, W_2} \sum_{t=1}^T \|X_t - W_2 W_1^T X_t\|^2$$

**PCA**:
$$\min_{\Lambda, F} \sum_{t=1}^T \|X_t - \Lambda F_t\|^2$$
subject to orthogonality constraints.

**Theorem**: With tied weights ($W_2 = W_1$) and orthogonality constraint, linear autoencoder = PCA.

In [ ]:
# Purpose: Implement linear autoencoder
# Key Concept: Linear autoencoder with tied weights equals PCA

class LinearAutoencoder(nn.Module):
    """
    Linear autoencoder with optional weight tying.
    
    Parameters
    ----------
    n_input : int
        Input dimension (N variables)
    n_latent : int
        Latent dimension (r factors)
    tied_weights : bool
        If True, decoder weights = encoder weights transpose
    """
    
    def __init__(self, n_input, n_latent, tied_weights=True):
        super(LinearAutoencoder, self).__init__()
        self.n_input = n_input
        self.n_latent = n_latent
        self.tied_weights = tied_weights
        
        # Encoder: X -> F
        self.encoder = nn.Linear(n_input, n_latent, bias=False)
        
        # Decoder: F -> X_hat (only if not tied)
        if not tied_weights:
            self.decoder = nn.Linear(n_latent, n_input, bias=False)
    
    def forward(self, x):
        """
        Forward pass: encode then decode.
        
        Returns
        -------
        x_reconstructed, latent_factors
        """
        # Encode
        latent = self.encoder(x)
        
        # Decode
        if self.tied_weights:
            # Tied: decoder = encoder^T
            x_reconstructed = torch.matmul(latent, self.encoder.weight)
        else:
            x_reconstructed = self.decoder(latent)
        
        return x_reconstructed, latent
    
    def get_loadings(self):
        """
        Extract encoder weights (factor loadings).
        
        Returns
        -------
        loadings : array, shape (n_input, n_latent)
        """
        return self.encoder.weight.data.cpu().numpy().T


def train_autoencoder(model, X, n_epochs=1000, lr=0.01, verbose=False):
    """
    Train autoencoder.
    
    Parameters
    ----------
    model : nn.Module
        Autoencoder model
    X : array, shape (T, N)
        Training data
    n_epochs : int
        Training epochs
    lr : float
        Learning rate
    verbose : bool
        Print progress
    
    Returns
    -------
    losses : list
        Training loss history
    """
    X_tensor = torch.FloatTensor(X).to(device)
    model = model.to(device)
    
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    losses = []
    
    for epoch in range(n_epochs):
        # Forward pass
        X_reconstructed, _ = model(X_tensor)
        loss = criterion(X_reconstructed, X_tensor)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        
        if verbose and epoch % 100 == 0:
            print(f"Epoch {epoch}: Loss = {loss.item():.6f}")
    
    return losses

print("LinearAutoencoder class implemented!")

### Verify PCA-Autoencoder Equivalence

In [ ]:
# Purpose: Compare PCA vs linear autoencoder empirically
# Key Concept: With tied weights, they should give similar results

# Generate factor model data
T, N, r = 200, 20, 3
F_true = np.random.randn(T, r)
Lambda_true = np.random.randn(N, r)
X_data = F_true @ Lambda_true.T + np.random.randn(T, N) * 0.5

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_data)

# 1. PCA
print("Fitting PCA...")
pca = PCA(n_components=r)
F_pca = pca.fit_transform(X_scaled)
Lambda_pca = pca.components_.T
X_reconstructed_pca = F_pca @ Lambda_pca.T
mse_pca = mean_squared_error(X_scaled, X_reconstructed_pca)

# 2. Linear Autoencoder (tied weights)
print("Training Linear Autoencoder (tied weights)...")
ae_tied = LinearAutoencoder(N, r, tied_weights=True)
losses_tied = train_autoencoder(ae_tied, X_scaled, n_epochs=1000, lr=0.01, verbose=False)

ae_tied.eval()
with torch.no_grad():
    X_tensor = torch.FloatTensor(X_scaled).to(device)
    X_reconstructed_ae, F_ae = ae_tied(X_tensor)
    X_reconstructed_ae = X_reconstructed_ae.cpu().numpy()
    F_ae = F_ae.cpu().numpy()

Lambda_ae = ae_tied.get_loadings()
mse_ae = mean_squared_error(X_scaled, X_reconstructed_ae)

# 3. Linear Autoencoder (separate weights)
print("Training Linear Autoencoder (separate weights)...")
ae_separate = LinearAutoencoder(N, r, tied_weights=False)
losses_separate = train_autoencoder(ae_separate, X_scaled, n_epochs=1000, lr=0.01, verbose=False)

ae_separate.eval()
with torch.no_grad():
    X_reconstructed_ae_sep, _ = ae_separate(X_tensor)
    X_reconstructed_ae_sep = X_reconstructed_ae_sep.cpu().numpy()

mse_ae_separate = mean_squared_error(X_scaled, X_reconstructed_ae_sep)

# Compare results
print("\n" + "="*70)
print("PCA vs LINEAR AUTOENCODER COMPARISON")
print("="*70)
print(f"PCA MSE: {mse_pca:.6f}")
print(f"Autoencoder (tied) MSE: {mse_ae:.6f}")
print(f"Autoencoder (separate) MSE: {mse_ae_separate:.6f}")

# Compare loadings (up to sign/rotation)
print(f"\nLoading correlation (PCA vs AE tied):")
for i in range(r):
    corr = np.corrcoef(Lambda_pca[:, i], Lambda_ae[:, i])[0, 1]
    print(f"  Factor {i+1}: {np.abs(corr):.4f}")

print("\n✓ Tied weights autoencoder closely matches PCA!")
print("="*70)

## 2. Deep Autoencoders for Nonlinear Factors

### Motivation

Linear factor models assume:
$$X_t = \Lambda F_t + e_t$$

Real data may have **nonlinear** relationships:
- Volatility (quadratic in returns)
- Regime-dependent loadings
- Interaction effects

**Deep autoencoders** can capture these nonlinearities:
$$F_t = \text{Encoder}(X_t; \theta_e)$$
$$\hat{X}_t = \text{Decoder}(F_t; \theta_d)$$

where encoders/decoders are neural networks.

In [ ]:
# Purpose: Implement deep autoencoder for nonlinear factors
# Key Concept: Multiple hidden layers capture nonlinearity

class DeepAutoencoder(nn.Module):
    """
    Deep autoencoder with nonlinear activations.
    
    Architecture:
        Encoder: X -> h1 -> h2 -> F (bottleneck)
        Decoder: F -> h3 -> h4 -> X_hat
    
    Parameters
    ----------
    n_input : int
        Input dimension
    n_latent : int
        Latent factor dimension
    hidden_dims : list
        Hidden layer dimensions [h1, h2]
    """
    
    def __init__(self, n_input, n_latent, hidden_dims=[64, 32]):
        super(DeepAutoencoder, self).__init__()
        self.n_input = n_input
        self.n_latent = n_latent
        
        # Encoder layers
        encoder_layers = []
        prev_dim = n_input
        for h_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.ReLU(),
                nn.BatchNorm1d(h_dim),
                nn.Dropout(0.2)
            ])
            prev_dim = h_dim
        
        encoder_layers.append(nn.Linear(prev_dim, n_latent))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Decoder layers (mirror of encoder)
        decoder_layers = []
        prev_dim = n_latent
        for h_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.ReLU(),
                nn.BatchNorm1d(h_dim),
                nn.Dropout(0.2)
            ])
            prev_dim = h_dim
        
        decoder_layers.append(nn.Linear(prev_dim, n_input))
        self.decoder = nn.Sequential(*decoder_layers)
    
    def forward(self, x):
        """
        Forward pass.
        
        Returns
        -------
        x_reconstructed, latent_factors
        """
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent
    
    def extract_factors(self, X):
        """
        Extract latent factors from data.
        
        Parameters
        ----------
        X : array, shape (T, N)
        
        Returns
        -------
        F : array, shape (T, r)
        """
        self.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(device)
            _, F = self.forward(X_tensor)
            return F.cpu().numpy()


def train_deep_autoencoder(model, X, n_epochs=200, batch_size=32, lr=0.001, verbose=True):
    """
    Train deep autoencoder with mini-batch SGD.
    
    Parameters
    ----------
    model : DeepAutoencoder
    X : array, shape (T, N)
    n_epochs : int
    batch_size : int
    lr : float
    verbose : bool
    
    Returns
    -------
    losses : list
    """
    X_tensor = torch.FloatTensor(X).to(device)
    dataset = TensorDataset(X_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss()
    
    losses = []
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        
        for batch in dataloader:
            X_batch = batch[0]
            
            # Forward
            X_reconstructed, _ = model(X_batch)
            loss = criterion(X_reconstructed, X_batch)
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        epoch_loss /= len(dataloader)
        losses.append(epoch_loss)
        
        if verbose and epoch % 20 == 0:
            print(f"Epoch {epoch}: Loss = {epoch_loss:.6f}")
    
    return losses

print("DeepAutoencoder class implemented!")

### Generate Nonlinear Data and Compare Methods

In [ ]:
# Purpose: Generate data with nonlinear factor structure
# Key Concept: Deep autoencoder should outperform PCA on nonlinear data

# Generate nonlinear data
T, N, r = 500, 30, 3
F_true = np.random.randn(T, r)

# Linear relationships (first half of variables)
Lambda_linear = np.random.randn(N // 2, r) * 0.5
X_linear = F_true @ Lambda_linear.T

# Nonlinear relationships (second half)
X_nonlinear = np.zeros((T, N // 2))
for i in range(N // 2):
    # Mix of linear and quadratic
    X_nonlinear[:, i] = (
        0.5 * F_true[:, i % r] +
        0.3 * F_true[:, (i+1) % r] ** 2 +
        np.random.randn(T) * 0.2
    )

X_nonlin_data = np.hstack([X_linear, X_nonlinear])
X_nonlin_data = (X_nonlin_data - X_nonlin_data.mean(axis=0)) / X_nonlin_data.std(axis=0)

print("="*70)
print("NONLINEAR DATA CHARACTERISTICS")
print("="*70)
print(f"Time periods (T): {T}")
print(f"Variables (N): {N}")
print(f"  Linear: {N//2}")
print(f"  Nonlinear: {N//2}")
print(f"True factors (r): {r}")
print("="*70)

In [ ]:
# Purpose: Compare PCA vs deep autoencoder on nonlinear data
# Key Concept: Nonlinear methods should improve on nonlinear data

# 1. PCA (linear)
print("\nFitting PCA (linear)...")
pca_nonlin = PCA(n_components=r)
F_pca_nonlin = pca_nonlin.fit_transform(X_nonlin_data)
X_recon_pca_nonlin = pca_nonlin.inverse_transform(F_pca_nonlin)
mse_pca_nonlin = mean_squared_error(X_nonlin_data, X_recon_pca_nonlin)

# 2. Deep Autoencoder (nonlinear)
print("\nTraining Deep Autoencoder (nonlinear)...")
deep_ae = DeepAutoencoder(N, r, hidden_dims=[32, 16])
losses_deep = train_deep_autoencoder(deep_ae, X_nonlin_data, n_epochs=100, 
                                     batch_size=32, lr=0.001, verbose=True)

F_deep = deep_ae.extract_factors(X_nonlin_data)
deep_ae.eval()
with torch.no_grad():
    X_tensor_nonlin = torch.FloatTensor(X_nonlin_data).to(device)
    X_recon_deep, _ = deep_ae(X_tensor_nonlin)
    X_recon_deep = X_recon_deep.cpu().numpy()

mse_deep = mean_squared_error(X_nonlin_data, X_recon_deep)

# Results
print("\n" + "="*70)
print("LINEAR vs NONLINEAR COMPARISON")
print("="*70)
print(f"PCA (linear) MSE: {mse_pca_nonlin:.6f}")
print(f"Deep AE (nonlinear) MSE: {mse_deep:.6f}")
improvement = (mse_pca_nonlin - mse_deep) / mse_pca_nonlin * 100
print(f"Improvement: {improvement:.2f}%")
print("="*70)

if improvement > 0:
    print("\n✓ Deep autoencoder captures nonlinear structure!")
else:
    print("\n⚠️  Deep autoencoder did not improve (may need more epochs/tuning)")

### Visualize Comparison

In [ ]:
# Purpose: Visualize linear vs nonlinear factor extraction
# Key Concept: Deep AE captures different factor structure

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Training loss
axes[0].plot(losses_deep, linewidth=2, color='steelblue')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('MSE Loss', fontsize=11)
axes[0].set_title('Deep Autoencoder Training', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot 2: Factor comparison (first factor)
axes[1].scatter(F_pca_nonlin[:, 0], F_deep[:, 0], alpha=0.5, s=20)
axes[1].set_xlabel('Linear Factor 1 (PCA)', fontsize=11)
axes[1].set_ylabel('Nonlinear Factor 1 (Deep AE)', fontsize=11)
axes[1].set_title('Factor Comparison', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Plot 3: Reconstruction error comparison
methods = ['PCA\n(Linear)', 'Deep AE\n(Nonlinear)']
mses = [mse_pca_nonlin, mse_deep]
colors = ['coral', 'seagreen']

bars = axes[2].bar(methods, mses, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
axes[2].set_ylabel('Reconstruction MSE', fontsize=11)
axes[2].set_title('Performance Comparison', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

# Highlight better method
best_idx = np.argmin(mses)
bars[best_idx].set_edgecolor('red')
bars[best_idx].set_linewidth(3)

plt.tight_layout()
plt.show()

## 3. Variational Autoencoders (VAE)

### Probabilistic Factor Models

**Traditional Factor Model**:
$$p(X | F, \Lambda, \Sigma) = \mathcal{N}(X; \Lambda F, \Sigma)$$
$$p(F) = \mathcal{N}(F; 0, I)$$

**Variational Autoencoder**:
$$p(X | F; \theta) = \mathcal{N}(X; \mu_\theta(F), \Sigma_\theta(F))$$
$$p(F) = \mathcal{N}(F; 0, I)$$
$$q(F | X; \phi) = \mathcal{N}(F; \mu_\phi(X), \Sigma_\phi(X))$$

**ELBO (Evidence Lower Bound)**:
$$\log p(X) \geq \mathbb{E}_{q(F|X)}[\log p(X|F)] - D_{KL}(q(F|X) \| p(F))$$

**Benefits**:
- Uncertainty quantification
- Regularization (KL term)
- Generative modeling

In [ ]:
# Purpose: Implement Variational Autoencoder
# Key Concept: VAE provides probabilistic factors with uncertainty

class VariationalAutoencoder(nn.Module):
    """
    Variational Autoencoder for probabilistic factor extraction.
    
    Learns distribution q(F|X) over latent factors.
    
    Parameters
    ----------
    n_input : int
        Input dimension
    n_latent : int
        Latent dimension
    hidden_dim : int
        Hidden layer dimension
    """
    
    def __init__(self, n_input, n_latent, hidden_dim=64):
        super(VariationalAutoencoder, self).__init__()
        self.n_input = n_input
        self.n_latent = n_latent
        
        # Encoder: X -> (mu, log_var)
        self.encoder_hidden = nn.Sequential(
            nn.Linear(n_input, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim)
        )
        self.fc_mu = nn.Linear(hidden_dim, n_latent)
        self.fc_logvar = nn.Linear(hidden_dim, n_latent)
        
        # Decoder: F -> X
        self.decoder = nn.Sequential(
            nn.Linear(n_latent, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, n_input)
        )
    
    def encode(self, x):
        """
        Encode X to latent distribution parameters.
        
        Returns
        -------
        mu, log_var : torch.Tensor
        """
        h = self.encoder_hidden(x)
        mu = self.fc_mu(h)
        log_var = self.fc_logvar(h)
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        """
        Reparameterization trick: F = mu + sigma * epsilon
        """
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        """Decode latent factors to reconstructed X."""
        return self.decoder(z)
    
    def forward(self, x):
        """
        Forward pass.
        
        Returns
        -------
        x_reconstructed, mu, log_var
        """
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        x_reconstructed = self.decode(z)
        return x_reconstructed, mu, log_var
    
    def loss_function(self, x_reconstructed, x, mu, log_var):
        """
        VAE loss = Reconstruction loss + KL divergence.
        
        Returns
        -------
        total_loss, recon_loss, kl_loss
        """
        # Reconstruction loss (MSE)
        recon_loss = nn.functional.mse_loss(x_reconstructed, x, reduction='sum')
        
        # KL divergence: D_KL(q(F|X) || N(0, I))
        kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
        
        return recon_loss + kl_loss, recon_loss, kl_loss


def train_vae(model, X, n_epochs=200, batch_size=32, lr=0.001, verbose=True):
    """Train Variational Autoencoder."""
    X_tensor = torch.FloatTensor(X).to(device)
    dataset = TensorDataset(X_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    losses = {'total': [], 'recon': [], 'kl': []}
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        epoch_recon = 0
        epoch_kl = 0
        
        for batch in dataloader:
            X_batch = batch[0]
            
            # Forward
            X_recon, mu, log_var = model(X_batch)
            loss, recon, kl = model.loss_function(X_recon, X_batch, mu, log_var)
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            epoch_recon += recon.item()
            epoch_kl += kl.item()
        
        n_batches = len(dataloader)
        losses['total'].append(epoch_loss / n_batches)
        losses['recon'].append(epoch_recon / n_batches)
        losses['kl'].append(epoch_kl / n_batches)
        
        if verbose and epoch % 20 == 0:
            print(f"Epoch {epoch}: Loss={losses['total'][-1]:.2f}, "
                  f"Recon={losses['recon'][-1]:.2f}, KL={losses['kl'][-1]:.2f}")
    
    return losses

print("VariationalAutoencoder class implemented!")

### Train VAE and Extract Factors with Uncertainty

In [ ]:
# Purpose: Train VAE for probabilistic factor extraction
# Key Concept: VAE provides uncertainty estimates for factors

print("Training Variational Autoencoder...")
vae = VariationalAutoencoder(N, r, hidden_dim=32)
vae_losses = train_vae(vae, X_nonlin_data, n_epochs=100, batch_size=32, lr=0.001, verbose=True)

# Extract factors with uncertainty
vae.eval()
with torch.no_grad():
    X_tensor_vae = torch.FloatTensor(X_nonlin_data).to(device)
    mu, log_var = vae.encode(X_tensor_vae)
    F_vae = mu.cpu().numpy()
    F_std = torch.exp(0.5 * log_var).cpu().numpy()

print("\n" + "="*70)
print("VAE FACTOR EXTRACTION WITH UNCERTAINTY")
print("="*70)
print(f"Extracted {r} latent factors with uncertainty estimates")
print(f"\nFirst observation:")
for i in range(r):
    print(f"  Factor {i+1}: {F_vae[0, i]:.3f} ± {F_std[0, i]:.3f}")
print("\nAverage uncertainty across all observations:")
for i in range(r):
    print(f"  Factor {i+1}: {F_std[:, i].mean():.3f}")
print("="*70)

### Visualize VAE Training

In [ ]:
# Purpose: Visualize VAE loss components
# Key Concept: VAE balances reconstruction and regularization

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss components
axes[0].plot(vae_losses['total'], label='Total Loss', linewidth=2)
axes[0].plot(vae_losses['recon'], label='Reconstruction', linewidth=2, linestyle='--')
axes[0].plot(vae_losses['kl'], label='KL Divergence', linewidth=2, linestyle=':')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].set_title('VAE Training Loss Components', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Factor uncertainty distribution
axes[1].hist(F_std[:, 0], bins=30, alpha=0.7, label='Factor 1', edgecolor='black')
axes[1].hist(F_std[:, 1], bins=30, alpha=0.7, label='Factor 2', edgecolor='black')
axes[1].set_xlabel('Standard Deviation', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Distribution of Factor Uncertainty', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Reconstruction loss: How well VAE reconstructs data")
print("- KL divergence: Regularization keeping factors close to N(0,I)")
print("- Higher uncertainty: Factor is less certain given observed data")

## 4. When to Use ML vs Traditional Factor Models

### Decision Framework

**Use Traditional Factor Models (PCA, MLE) When:**
- Interpretation matters (economic meaning)
- Statistical inference needed (tests, CIs)
- Small/moderate samples (T < 1000)
- Linear relationships suffice
- Transparency required

**Use Neural Network Autoencoders When:**
- Large datasets (T > 10,000)
- Nonlinear structure evident
- Prediction primary goal
- Rich features (text, images)
- Computational resources available

**Use VAE When:**
- Uncertainty quantification needed
- Generative modeling desired
- Regularization important
- Probabilistic framework valuable

## 5. Exercises

### Exercise 5.1: Verify PCA-Autoencoder Equivalence

**Task:** Show that as you increase training epochs for the tied-weight autoencoder, loadings converge to PCA loadings.

In [ ]:
# Exercise 5.1: Convergence analysis

# YOUR CODE HERE
# ---------------
# 1. Train tied-weight autoencoder for different n_epochs: [100, 500, 1000, 5000]
# 2. For each, compute correlation between AE loadings and PCA loadings
# 3. Plot convergence: correlation vs epochs

# AUTO-GRADED TESTS
def test_exercise_5_1():
    print("Implement Exercise 5.1: Verify convergence to PCA")
    print("Expected: Plot showing loading correlation approaching 1.0")

test_exercise_5_1()

### Exercise 5.2: Denoising Autoencoder

**Task:** Implement a denoising autoencoder that adds noise to inputs during training. Compare reconstruction to standard autoencoder.

In [ ]:
# Exercise 5.2: Denoising autoencoder

# YOUR CODE HERE
# ---------------
# 1. Modify training loop to add Gaussian noise to inputs
# 2. Train denoising autoencoder
# 3. Compare reconstruction MSE on clean test data

# AUTO-GRADED TESTS
def test_exercise_5_2():
    print("Implement Exercise 5.2: Denoising autoencoder")
    print("Expected: Comparison showing denoising AE is more robust to noise")

test_exercise_5_2()

### Exercise 5.3: Hybrid Model

**Task:** Create a hybrid model that extracts PCA factors, then uses a neural network for forecasting. Compare to pure PCA and pure deep autoencoder.

In [ ]:
# Exercise 5.3: Hybrid PCA + Neural Network

# YOUR CODE HERE
# ---------------
# 1. Extract factors with PCA
# 2. Build neural network for forecasting from factors
# 3. Compare forecast MSE to pure PCA regression and end-to-end deep AE

# AUTO-GRADED TESTS
def test_exercise_5_3():
    print("Implement Exercise 5.3: Hybrid model")
    print("Expected: Performance comparison of 3 approaches")

test_exercise_5_3()

## 6. Summary

### Key Takeaways

1. **Linear autoencoders with tied weights are mathematically equivalent to PCA** - both minimize reconstruction error

2. **Deep autoencoders generalize to nonlinear factor structures** but sacrifice interpretability and statistical theory

3. **VAEs add probabilistic interpretation** with uncertainty quantification via ELBO objective

4. **Trade-offs are fundamental**: interpretability/theory vs flexibility/prediction

5. **Hybrid approaches** combining traditional econometrics with machine learning often perform best

### Course Conclusion

You've completed a comprehensive journey through dynamic factor models:

- **Static factors** (PCA, maximum likelihood)
- **Dynamic factors** (state-space, Kalman filtering)
- **Sparse methods** (LASSO, targeted predictors)
- **Advanced topics** (time-varying parameters, ML connections)

### Key Skills Acquired

- Extract interpretable factors from high-dimensional data
- Implement Kalman filter and EM algorithm
- Build forecasting systems with factors
- Select relevant variables in high dimensions
- Apply modern ML methods appropriately
- Critically evaluate method trade-offs

### Next Steps

- Apply these methods to your domain
- Read cutting-edge research papers
- Contribute to open-source libraries
- Develop novel extensions

**The field continues to evolve rapidly. Stay curious, rigorous, and always validate with out-of-sample data!**

---

## Additional Resources

**Key Papers:**
- Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*. MIT Press.
- Kingma, D.P. & Welling, M. (2014). "Auto-encoding variational Bayes." *ICLR*.
- Gu, S., Kelly, B. & Xiu, D. (2020). "Empirical asset pricing via machine learning." *RFS*.
- Chen, L., Pelger, M. & Zhu, J. (2023). "Deep learning in asset pricing." *Management Science*.

**Software:**
- PyTorch documentation: [pytorch.org](https://pytorch.org)
- Scikit-learn: [scikit-learn.org](https://scikit-learn.org)